In [0]:
# Databricks notebook source
# tutorial_name: 01-Day7-Structured-Streaming-Delta
# notebookName: 01-Day7-Structured-Streaming-Delta
# COMMAND ----------

# DBTITLE 1,Paths
notepoint_rel = "hands-on/day-07/notebooks/01-Day7-Structured-Streaming-Delta.ipynb"
BASE_PATH = "abfss://atininput@sadbtrng19032026.dfs.core.windows.net/data/"
STUDENT_ID = "u25"  # Change to your assigned u01–u16
DAY5 = BASE_PATH + "/day5"
P_BASIC = DAY5 + "/delta/flight_summary_basic"
DAY7_PREFIX = f"{BASE_PATH}day07-{STUDENT_ID}"
STREAM_OUT = f"{DAY7_PREFIX}/stream/rate_to_delta"
STREAM_CP = f"{DAY7_PREFIX}/checkpoints/rate_to_delta"
# COMMAND ----------

# DBTITLE 1,Prerequisite check
try:
    spark.read.format("delta").load(P_BASIC).limit(1).collect()
    print("✓ Day 5 tables found (P_BASIC)")
except Exception as e:
    print(f"✗ Day 5 tables not found: {e}")
    print("Please complete Day 5 notebook 01 first.")

print("STREAM_OUT:", STREAM_OUT)
print("STREAM_CP:", STREAM_CP)
# COMMAND ----------


## Day 7 — Item 19: Structured Streaming to Delta

This notebook uses a **rate** source (built-in test stream), writes **append** output to Delta under your `day07-{STUDENT_ID}` prefix, and uses a **checkpoint** location. Then you enable **Change Data Feed (CDF)** and read changes.

**Agenda:** root `README.md` — Day 7, item 19.


### Lab 1 — Micro-batch stream: rate → Delta

`availableNow=True` processes available data then stops (good for class demos). For a long-running job, use `.trigger(processingTime="10 seconds")` instead.


In [0]:
from pyspark.sql.functions import col

rate_df = (
    spark.readStream.format("rate")
    .option("rowsPerSecond", 3)
    .option("numPartitions", 1)
    .load()
)
stream_df = rate_df.withColumn("mod100", col("value") % 100)

try:
    for s in spark.streams.active:
        if getattr(s, "name", None) == "rate_to_delta":
            s.stop()
except Exception:
    pass

q = (
    stream_df.writeStream.format("delta")
    .outputMode("append")
    .option("checkpointLocation", STREAM_CP)
    .queryName("rate_to_delta")
    .trigger(availableNow=True)
    .start(STREAM_OUT)
)
q.awaitTermination()
print("Wrote micro-batches to:", STREAM_OUT)


### Lab 2 — Read the sink in batch mode


In [0]:
batch = spark.read.format("delta").load(STREAM_OUT)
print(batch.count())
batch.orderBy("timestamp").show(5, truncate=False)


### Lab 3 — Enable Change Data Feed (CDF)

After CDF is on, further writes produce change metadata you can read with `readChangeFeed`.


In [0]:
spark.sql(
    f"ALTER TABLE delta.`{STREAM_OUT}` SET TBLPROPERTIES ('delta.enableChangeDataFeed' = 'true')"
)
print("CDF enabled on sink table.")


### Lab 4 — Append more rows (generates insert changes)


In [0]:
q2 = (
    stream_df.writeStream.format("delta")
    .outputMode("append")
    .option("checkpointLocation", STREAM_CP)
    .queryName("rate_to_delta")
    .trigger(availableNow=True)
    .start(STREAM_OUT)
)
q2.awaitTermination()
print("Appended another available-now batch.")


### Lab 5 — Read the change feed


In [0]:
cdf_df = (
    spark.read.format("delta")
    .option("readChangeFeed", "true")
    .option("startingVersion", 0)
    .load(STREAM_OUT)
)
cdf_df.orderBy("_commit_version", "_commit_timestamp").show(20, truncate=False)


### Optional — Clean up (your prefix only)

Uncomment to remove checkpoint and stream output under your Day 7 prefix when you are done experimenting.


In [0]:
# dbutils.fs.rm(DAY7_PREFIX, recurse=True)
